# Week 07 Puzzles — PsychoPy Experiment Design

> **NS5116 Computational Neuroscience — Spring 2026**

This notebook is divided into two parts:

- **Part 1 — Guided Practice (Puzzles 1–10):** Each puzzle includes a worked solution. Study them, run the cells, and modify the code to test your understanding.
- **Part 2 — Independent Practice (Puzzles 11–20):** Write your own solution in the empty code cells. There is no single correct answer — focus on clarity, correctness, and Pythonic style.

> **Note:** These puzzles focus on the *data and logic* side of experiment design — building trial lists, saving data, computing summaries — so they run in a Jupyter notebook without a PsychoPy window. The actual stimulus presentation code is practised in the weekly assignment.

---

## Part 1 — Guided Practice (with solutions)

### Puzzle 1 — Build a Trial List from Conditions

Define the four unique cue×target conditions for a Posner task as a list of dicts. Replicate each condition 5 times to create a 20-trial list.

Print the total number of trials and the first 3 trials.

In [ ]:
conditions = [
    {"cue": "left",  "target": "left",  "validity": "valid"},
    {"cue": "right", "target": "right", "validity": "valid"},
    {"cue": "left",  "target": "right", "validity": "invalid"},
    {"cue": "right", "target": "left",  "validity": "invalid"},
]

n_reps = 5
trial_list = conditions * n_reps

print(f"Total trials: {len(trial_list)}")
for t in trial_list[:3]:
    print(t)

### Puzzle 2 — Shuffle and Number Trials

Take the 20-trial list from Puzzle 1, shuffle it with a fixed seed for reproducibility, then add a `trial_num` field (1-indexed) to each trial dict.

Print trials 1–5 to verify the numbering and randomization.

In [ ]:
import random
import copy

conditions = [
    {"cue": "left",  "target": "left",  "validity": "valid"},
    {"cue": "right", "target": "right", "validity": "valid"},
    {"cue": "left",  "target": "right", "validity": "invalid"},
    {"cue": "right", "target": "left",  "validity": "invalid"},
]

trial_list = [copy.deepcopy(c) for c in conditions * 5]

random.seed(42)
random.shuffle(trial_list)

for i, trial in enumerate(trial_list):
    trial["trial_num"] = i + 1

for t in trial_list[:5]:
    print(f"Trial {t['trial_num']:2d}: cue={t['cue']:5s}  target={t['target']:5s}  validity={t['validity']}")

### Puzzle 3 — Validate Condition Balance

After shuffling, it is good practice to verify that the condition counts are correct.

Count how many trials are valid vs. invalid. Print the counts and percentages. Verify that valid trials make up 50% of the total (since we have 2 valid + 2 invalid condition types).

In [ ]:
valid_count   = sum(1 for t in trial_list if t["validity"] == "valid")
invalid_count = sum(1 for t in trial_list if t["validity"] == "invalid")
total = len(trial_list)

print(f"Valid:   {valid_count}/{total} ({valid_count/total:.0%})")
print(f"Invalid: {invalid_count}/{total} ({invalid_count/total:.0%})")
print(f"Balance check: {'✓ balanced' if valid_count == invalid_count else '✗ UNBALANCED'}")

### Puzzle 4 — Build a Participant Info Dictionary

In PsychoPy, `gui.DlgFromDict` creates a dialog from a dictionary. Simulate this by creating a participant info dict with fields: `Participant ID`, `Age`, `Session`, and `Date`.

Auto-fill the `Date` field using `datetime.date.today()`. Print the info dict.

In [ ]:
import datetime

info = {
    "Participant ID": "P01",
    "Age": 22,
    "Session": 1,
    "Date": datetime.date.today().isoformat(),
}

print("Participant info:")
for key, value in info.items():
    print(f"  {key:<16}: {value}")

### Puzzle 5 — Simulate a Single Trial and Record the Result

Write a function `simulate_trial(params)` that simulates one trial:
- Generate a random RT from `normal(450, 80)` ms
- Determine `correct` by randomly choosing True/False (80% correct)
- Merge the condition dict with response fields using `{**params, ...}`

Call it on a sample trial and print the result.

In [ ]:
import random

def simulate_trial(params, rng):
    """Simulate one trial: generate RT and accuracy.

    Args:
        params (dict): Trial parameters (cue, target, validity, trial_num).
        rng (random.Random): Random number generator.

    Returns:
        dict: Merged dict with response, rt, and correct fields added.
    """
    rt = rng.gauss(450, 80)
    correct = rng.random() < 0.80
    response = params["target"] if correct else ("left" if params["target"] == "right" else "right")

    return {**params, "response": response, "rt": round(rt, 2), "correct": correct}


rng = random.Random(42)
sample_trial = {"cue": "left", "target": "left", "validity": "valid", "trial_num": 1}
result = simulate_trial(sample_trial, rng)

for key, value in result.items():
    print(f"  {key:<12}: {value}")

### Puzzle 6 — Run a Full Block and Collect Results

Write a function `run_block(trial_list, rng)` that loops over a list of trial dicts, calls `simulate_trial()` on each, and returns a list of result dicts.

Run it on the first 10 trials and print the number of correct responses.

In [ ]:
def run_block(trials, rng):
    """Simulate an entire block of trials.

    Args:
        trials (list[dict]): List of trial parameter dicts.
        rng (random.Random): Random number generator.

    Returns:
        list[dict]: Results with response data merged in.
    """
    return [simulate_trial(t, rng) for t in trials]


rng = random.Random(42)
block_results = run_block(trial_list[:10], rng)

n_correct = sum(r["correct"] for r in block_results)
print(f"Correct: {n_correct}/{len(block_results)} ({n_correct/len(block_results):.0%})")
print(f"\nFirst 3 results:")
for r in block_results[:3]:
    print(f"  Trial {r['trial_num']:2d}: RT={r['rt']:6.1f} ms  correct={r['correct']}")

### Puzzle 7 — Split Trials into Blocks

Given 20 trials, split them into 2 blocks of 10 trials each.  
Use list slicing based on `trials_per_block = len(all_trials) // n_blocks`.

Print the trial numbers in each block to verify the split.

In [ ]:
n_blocks = 2
trials_per_block = len(trial_list) // n_blocks

for b in range(n_blocks):
    start = b * trials_per_block
    end   = start + trials_per_block
    block_trials = trial_list[start:end]
    trial_nums = [t["trial_num"] for t in block_trials]
    print(f"Block {b+1}: trials {trial_nums}")

### Puzzle 8 — Save Results to CSV with `csv.DictWriter`

Write a function `save_results(filepath, results)` that saves a list of result dicts to a CSV file using `csv.DictWriter`.

Save 10 simulated results, then read the file back and print its contents.

In [ ]:
import csv
import os

def save_results(filepath, results):
    """Save a list of trial-result dicts to CSV.

    Args:
        filepath (str): Output CSV path.
        results (list[dict]): Trial result dicts.
    """
    fieldnames = list(results[0].keys())
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)


rng = random.Random(42)
results = run_block(trial_list[:10], rng)

CSV_FILE = "posner_results.csv"
save_results(CSV_FILE, results)

# Read back and verify
with open(CSV_FILE, encoding="utf-8") as f:
    for line in f:
        print(line, end="")

# Clean up
os.remove(CSV_FILE)

### Puzzle 9 — Compute Per-Condition Summary Statistics

Given a list of result dicts, compute mean RT and accuracy **per validity condition** (valid vs. invalid).  
Only include correct trials when computing mean RT (standard practice in cognitive experiments).

In [ ]:
rng = random.Random(42)
all_results = run_block(trial_list, rng)

for condition in ["valid", "invalid"]:
    subset = [r for r in all_results if r["validity"] == condition]
    correct_rts = [r["rt"] for r in subset if r["correct"]]
    accuracy = sum(r["correct"] for r in subset) / len(subset)

    if correct_rts:
        mean_rt = sum(correct_rts) / len(correct_rts)
    else:
        mean_rt = float("nan")

    print(f"{condition:8s}  accuracy={accuracy:.0%}  mean RT (correct)={mean_rt:.1f} ms  n={len(subset)}")

### Puzzle 10 — Generate the Data Filename from Participant Info

Write a function `make_filename(info)` that produces a standardized data filename from the participant info dict.

Format: `data/<Participant ID>_session<Session>.csv`

The function should also create the `data/` directory if it does not exist.  
Test it with sample info and verify the filename format.

In [ ]:
import os

def make_filename(info):
    """Generate a standardized data filename from participant info.

    Args:
        info (dict): Must contain 'Participant ID' and 'Session'.

    Returns:
        str: Path like 'data/P01_session1.csv'
    """
    os.makedirs("data", exist_ok=True)
    return f"data/{info['Participant ID']}_session{info['Session']}.csv"


test_info = {"Participant ID": "P03", "Session": 2}
filename = make_filename(test_info)
print(f"Filename: {filename}")
print(f"Directory exists: {os.path.isdir('data')}")

# Clean up
if os.path.isdir("data") and not os.listdir("data"):
    os.rmdir("data")

---

## Part 2 — Independent Practice (write your own solutions)

The cells below contain only problem descriptions. Write your solution in the provided code cell.  
There may be more than one correct approach — prioritise readability and correctness.

### Puzzle 11 — Build a Trial List with Unequal Proportions

Create a 40-trial Posner task where **80% of trials are valid** and **20% are invalid**, matching the standard cueing ratio.

Hints:
- 2 valid conditions × 8 reps = 16 valid trials each, so 32 total valid
- 2 invalid conditions × 2 reps = 4 invalid trials each, so 8 total invalid

Shuffle and verify: print the proportion of valid trials.

In [ ]:
# Your solution here


### Puzzle 12 — Add Block Numbers to Results

Given a list of 20 result dicts and 2 blocks of 10 trials each, add a `"block"` key to each result dict indicating which block it belongs to.

Use the block-splitting pattern from Puzzle 7. Print the block number and trial number for each trial.

In [ ]:
# Your solution here


### Puzzle 13 — Detect Anticipation Responses

Write a function `flag_anticipations(results, threshold=100)` that marks trials with RT < threshold ms as anticipation responses by setting `result["anticipation"] = True`.

Test on a list of results where you manually set some RTs below 100 ms. Print the number of flagged trials.

In [ ]:
# Your solution here


### Puzzle 14 — Compute the Cueing Effect

The **cueing effect** is the difference in mean RT between invalid and valid trials (invalid RT − valid RT). A positive value means valid cues speed up responses.

Write a function `cueing_effect(results)` that returns this difference.  
Use only correct trials for the calculation. Print the cueing effect in ms.

In [ ]:
# Your solution here


### Puzzle 15 — Generate a Rest Screen Message

Write a function `rest_message(block_num, n_blocks, accuracy)` that returns a formatted multi-line string suitable for a rest screen:

```
Block 1 of 3 complete.
Accuracy so far: 85%

Take a short break.
Press SPACE when ready.
```

Test it with sample values and print the output.

In [ ]:
# Your solution here


### Puzzle 16 — Validate Participant Info

Write a function `validate_info(info)` that checks:
1. `Participant ID` is a non-empty string
2. `Age` is an integer between 18 and 99
3. `Session` is a positive integer

Return a list of error messages (empty list if all checks pass).  
Test with both valid and invalid info dicts.

In [ ]:
# Your solution here


### Puzzle 17 — Merge Participant Info into Results

Write a function `add_participant_info(results, info)` that adds `subject_id` and `session` fields to every result dict, extracted from the participant info dict.

This is important for combining data across participants later.  
Test on a small results list and print one result to verify the added fields.

In [ ]:
# Your solution here


### Puzzle 18 — Compute Per-Block Accuracy Trend

Given results from a 40-trial experiment (4 blocks of 10), compute the accuracy for each block separately. Does accuracy improve across blocks (practice effect)?

Simulate results with slightly increasing accuracy (e.g., 70%, 75%, 80%, 85%) and verify your function detects the trend.

In [ ]:
# Your solution here


### Puzzle 19 — Read Results Back from CSV and Recompute

Write code that:
1. Creates a mock CSV file with columns: `trial_num`, `validity`, `response`, `rt`, `correct`
2. Reads it back with `csv.DictReader`
3. Converts `rt` to float and `correct` to bool
4. Recomputes mean RT per validity condition

This tests the full write → read → analyze cycle.

In [ ]:
# Your solution here


### Puzzle 20 — Full Mini-Experiment Pipeline *(Bonus)*

Put it all together. Write a single script that:
1. Creates participant info (simulated)
2. Builds a 40-trial list (80% valid, 20% invalid)
3. Shuffles trials and adds trial numbers
4. Splits into 2 blocks of 20
5. Simulates responses for all trials
6. Adds participant info and block numbers to results
7. Saves to CSV
8. Reads the CSV back and prints a summary (mean RT and accuracy per validity × block)
9. Cleans up the CSV file

Use functions from the earlier puzzles.

In [ ]:
# Your solution here
